# Hate Speech Classification with LLMs

This notebook explores using Large Language Models (LLMs) for hate speech classification on the HateXplain dataset.
We will use a prompt engineering approach to classify text into "hatespeech", "normal", or "offensive".

In [ ]:
%pip install -q openai "datasets<3.0" "pyarrow<20" scikit-learn pandas python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Shari\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# Install CPU-only PyTorch to avoid DLL errors on machines without NVIDIA GPUs
%pip install --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached https://download.pytorch.org/whl/fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/mpmath-1.3.0-py3-none-any.whl (536 kB)
  Using cached https://download.pytorch.org/whl/MarkupSafe-2.1.5-cp311-cp311-win_amd64.whl (17 kB)
Using cached https://download.pytorch.org/whl/cpu/to

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.1.0 requires pyjwt>=2.10.0, but you have pyjwt 2.8.0 which is incompatible.
fastmcp 2.12.5 requires python-dotenv>=1.1.0, but you have python-dotenv 1.0.0 which is incompatible.
postgrest 0.13.2 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.
storage3 0.6.1 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.
supabase 2.0.0 requires httpx<0.25.0,>=0.24.0, but you have httpx 0.28.1 which is incompatible.
supafunc 0.3.3 requires httpx<0.26,>=0.24, but you have httpx 0.28.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Shari\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os
import json
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI Client for Local Ollama
# Ollama provides an OpenAI-compatible API at /v1
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # API key is required but ignored by Ollama
)

print("Client initialized for local Ollama.")

Client initialized for local Ollama.


In [7]:
# Data Processing Helpers (Reused from BERT Notebook)

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    if ann is None: return []
    if isinstance(ann, str):
        try: ann = json.loads(ann)
        except: return []
    if isinstance(ann, list):
        if len(ann) == 0: return []
        if isinstance(ann[0], dict): return ann
        out = []
        for x in ann:
            try: out.append({"label": int(x), "target": []})
            except: pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        n = len(labels)
        out = []
        for i in range(n):
            try: lab = int(labels[i])
            except: continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    if rats is None: return []
    if isinstance(rats, str):
        try: rats = json.loads(rats)
        except: return []
    if isinstance(rats, list):
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats: return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex: return str(ex["text"])
    if "cleaned_text" in ex: return str(ex["cleaned_text"])
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,
        "targets": targets,
    }

In [15]:
# Load and Prepare Data
print("Loading HateXplain dataset...")
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# Normalize
# Removed num_proc=2 to avoid pickling errors on Windows where helper functions aren't found in worker processes
ds = ds.map(normalize_example)

# Keep only necessary columns
to_keep = ["text_norm", "label_norm"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Filter empty
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

Loading HateXplain dataset...


In [18]:
# Select Test Set
test_ds = ds["test"]
print(f"Full Test set size: {len(test_ds)}")

# Select Subset of Test Set (e.g. first 20 examples)
# We use .select() to keep it as a Dataset object
test_ds = ds["test"].select(range(20))
print(f"Subset test set size: {len(test_ds)}")

Full Test set size: 1924
Subset test set size: 20


In [21]:
def classify_text(text, model="gpt-oss:20b"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull llama3`
    """
    system_prompt = (
        "You are an expert content moderator. "
        "Classify the following text into one of these three categories: 'hatespeech', 'normal', 'offensive'. "
        "Provide ONLY the category name as your response, in lowercase."
    )
    
    user_prompt = f"Text: {text}"
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=10
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

# Test the function
# Ensure you have the model running in Ollama
sample_text = test_ds[0]['text_norm']
print(f"Sample Text: {sample_text}")
print(f"Prediction: {classify_text(sample_text)}")

Sample Text: <user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children
Prediction: 
Prediction: 


In [19]:
# Run Inference on Test Set
predictions = []
ground_truth = []
texts = []

print("Running inference...")
for example in tqdm(test_ds):
    text = example['text_norm']
    label_id = example['label_norm']
    true_label = ID2NAME[label_id]
    
    pred_label = classify_text(text)
    
    # Handle potential API errors or unexpected output
    if pred_label not in NAME2ID:
        # Simple fallback or mapping if the LLM is chatty (though prompt says ONLY label)
        if "hate" in pred_label: pred_label = "hatespeech"
        elif "offen" in pred_label: pred_label = "offensive"
        elif "norm" in pred_label: pred_label = "normal"
        else: pred_label = "normal" # Default fallback
        
    predictions.append(pred_label)
    ground_truth.append(true_label)
    texts.append(text)

# Create a DataFrame for results
results_df = pd.DataFrame({
    "text": texts,
    "ground_truth": ground_truth,
    "prediction": predictions
})

results_df.head()

Running inference...


100%|██████████| 20/20 [02:52<00:00,  8.61s/it]



,text,ground_truth,prediction
0,<user> men can not be raped can not be abused ...,normal,offensive
1,<user> you are missing an essential premise wh...,normal,offensive
2,<user> <user> why are you repeating yourself a...,offensive,offensive
3,<user> <user> well she ’ muslim so of course s...,hatespeech,offensive
4,<user> lol not me i don ’ t deal with bms not ...,offensive,offensive


In [22]:
# Display full results with complete text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Complete Results - Full Text View:\n")
print(results_df.to_string(index=True))

Complete Results - Full Text View:

                                                                                                                                                                                                                                                                                             text ground_truth  prediction
0                                                                                                           <user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children       normal   offensive
1                                                                                                                                                                                        <user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass       normal   offensive
2                  

In [20]:
# Evaluation
print("\nClassification Report:")
print(classification_report(ground_truth, predictions, target_names=["hatespeech", "normal", "offensive"], labels=["hatespeech", "normal", "offensive"]))

print("\nConfusion Matrix:")
print(confusion_matrix(ground_truth, predictions, labels=["hatespeech", "normal", "offensive"]))


Classification Report:
              precision    recall  f1-score   support

  hatespeech       0.50      0.14      0.22         7
      normal       0.50      0.14      0.22         7
   offensive       0.25      0.67      0.36         6

    accuracy                           0.30        20
   macro avg       0.42      0.32      0.27        20
weighted avg       0.42      0.30      0.26        20


Confusion Matrix:
[[1 0 6]
 [0 1 6]
 [1 1 4]]
